In [16]:
import getpass
import numpy as np
import matplotlib.pyplot as plt
import rasterio
import openeo

from datetime import date

In [17]:

CLIENT_ID = getpass.getpass("Introduce tu CLIENT ID de Sentinel Hub: ")
CLIENT_SECRET = getpass.getpass("Introduce tu CLIENT SECRET de Sentinel Hub: ")

conn = openeo.connect("https://openeo.sentinel-hub.com/production")
conn.authenticate_basic(CLIENT_ID, CLIENT_SECRET)



Introduce tu CLIENT ID de Sentinel Hub:  ········
Introduce tu CLIENT SECRET de Sentinel Hub:  ········


<Connection to 'https://openeo.sentinel-hub.com/production/' with BasicBearerAuth>

In [18]:
lago_atitlan = {
    "west": -91.326256,
    "east": -91.07151,
    "south": 14.5948,
    "north": 14.750979
}

lago_amatitlan = {
    "west": -90.638065,
    "east": -90.512924,
    "south": 14.412347,
    "north": 14.493799
}

start_date = "2023-01-01"
end_date = "2023-06-30"
# Aqui arriba se definieron lo de la observacion de los 6 meses que nos solicitaron


cube_atitlan = conn.load_collection(
    "SENTINEL2_L2A_SENTINELHUB",
    spatial_extent=lago_atitlan,
    temporal_extent=[start_date, end_date],
    bands=["B04", "B05"]
)

cube_amatitlan = conn.load_collection(
    "SENTINEL2_L2A_SENTINELHUB",
    spatial_extent=lago_amatitlan,
    temporal_extent=[start_date, end_date],
    bands=["B04", "B05"]
)

In [21]:
def ndci(cube):
    b5 = cube.band("B05")
    b4 = cube.band("B04")
    return (b5 - b4) / (b5 + b4)

# Reducir a valor máximo en el tiempo
ndci_atitlan = ndci(cube_atitlan).max_time()
ndci_amatitlan = ndci(cube_amatitlan).max_time()

# Descargar resultados con nombres personalizados
job_atitlan = ndci_atitlan.create_job(out_format="GTiff")
job_atitlan.start_and_wait()
job_atitlan.download_results(target="NDCI_Atitlan_Max6m.tif")

job_amatitlan = ndci_amatitlan.create_job(out_format="GTiff")
job_amatitlan.start_and_wait()
job_amatitlan.download_results(target="NDCI_Amatitlan_Max6m.tif")

# Función para abrir GeoTIFF como array
def load_tif_as_array(path):
    with rasterio.open(path) as src:
        arr = src.read(1)
    return arr

# Cargar arrays
arr_atitlan = load_tif_as_array("NDCI_Atitlan_Max6m.tif")
arr_amatitlan = load_tif_as_array("NDCI_Amatitlan_Max6m.tif")

print("Atitlán array shape:", arr_atitlan.shape)
print("Amatitlán array shape:", arr_amatitlan.shape)

OpenEoApiError: [500] Internal: Server error: Failed to download from:
https://services.sentinel-hub.com/api/v1/batch/process
with HTTPError:
403 Client Error: Forbidden for url: https://services.sentinel-hub.com/api/v1/batch/process
Server response: "{"status": 403, "reason": "Forbidden", "message": "You are not authorized to perform this action.", "code": "COMMON_INSUFFICIENT_PERMISSIONS"}"